## Creando una Vista Hercodeada

Views - Vistas

- Vistas basadas en funciones
- Vistas basadas en clases

## Como crear una aplicacion en Django

 Python manage.py startapp <nombre de la app>
 python manage.py startapp ecommerce

 Convenciones y buenas practicas

 -funciones se nombran con minusculas
 - clases se nombra capital

## ecommerce/Views.py

In [ ]:
from django.shortcuts import render
from django.http import HttpResponse

def home(request):
    return HttpResponse("Hola Mundo")

## ecommerce/urls.py

In [ ]:
from django.conf import settings
from django.contrib import admin
from django.urls import include, path
from ecommerce import views


urlpatterns = [
    path("", views.home, name="home"),
]

## config/urls.py

In [ ]:
from django.conf import settings
from django.contrib import admin
from django.urls import include, path



urlpatterns = [
    path("up/", include("up.urls")),
    path("", include("pages.urls")),
    path("ecommerce/", include("ecommerce.urls")),
    path("admin/", admin.site.urls),
]

In [ ]:
from django.shortcuts import render
from django.http import HttpResponse

# Create your views here.
def home(request):
    html = """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <style>
            h1{
                color: blue;
            }
        </style>
    </head>
    <body>
        <h1>Hola Mundo</h1>
    </body>
    </html>
    """
    return HttpResponse(html)

## CRUD y Vistas

Forma dinamica de usar las vistas
CRUD - Create, Retrive, Update, Delete
- Crear modelos
- Agregar la app a INSTALLED_APPS
- Crear y aplicar migraciones
- Agregar admin

## ecommerce/Models.py

In [ ]:
from django.db import models

# Create your models here.
class ProductModel(models.Model):
    title = models.TextField()
    price = models.FloatField()

## config/settings.py

In [ ]:
INSTALLED_APPS = [
    "pages.apps.PagesConfig",
    "ecommerce.apps.EcommerceConfig",
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
]

## Como correr migraciones

- python manage.py makemigrations
- python manage.py migrate

## Como registar modelos en el admin

In [ ]:
from django.contrib import admin

# Register your models here.
from .models import ProductModel
admin.site.register(ProductModel)

## Como crear un superusuario para el admin

- python manage.py createsuperuser

## Tipos basicos de vistas

- list View: donde ves los usuarios creados
- Create View: donde insertas nuevos datos
- Retrieve and Update View
- Delete View

## Usando Templates (sin contexto)
### ecommerce/views.py

In [ ]:
from django.shortcuts import render
from .models import ProductModel

def product_model_list_view(request):
    qs = ProductModel.objects.all()
    print(qs)
    template = "ecommerce/list-view.html"
    context = {}
    return render(request, template, context)

## Templates/ecommerce/list-view.html

In [ ]:
<h1> Desde mi template </h1>

## Usando Templates (Con contexto)
### ecommerce/views.py

In [ ]:
from django.shortcuts import render
from .models import ProductModel

def product_model_list_view(request):
    qs = ProductModel.objects.all()
    print(qs)
    template = "ecommerce/list-view.html"
    context = {
        "products" : qs
    }
    return render(request, template, context)

In [ ]:
<h1>Dede mi template</h1>

<ul>
    {% for product in products %}
        <li><strong>{{ product.title }}</strong> - {{ product.price }}</li>
    {% endfor %}
</ul>

## Protegiendo los endpoints

### Opcion 1 - Usando request.user

In [ ]:
from django.shortcuts import render
from .models import ProductModel

def product_model_list_view(request):
    qs = ProductModel.objects.all()
    print(qs)

    context = {
        "products" : qs,
        "user" : request.user
    }

    if request.user.is_authenticated:
        template = "ecommerce/list-view.html"
    else:
        template = "ecommerce/public-list-view.html"

    return render(request, template, context)

### Opcion 1 - decorador login required

In [ ]:
from django.shortcuts import render
from .models import ProductModel
from django.contrib.auth.decorators import login_required



@login_required(login_url="/admin/login/")
def product_model_list_view(request):
    qs = ProductModel.objects.all()
    print(qs)

    context = {
        "products" : qs,
        "user" : request.user
    }

    template = "ecommerce/list-view.html"

    return render(request, template, context)

## Detail View

In [ ]:
from django.shortcuts import render, get_object_or_404
def product_model_detail_view(request, id):
    instance = get_object_or_404(ProductModel, id=id)

    context = {
        "product" : instance,
    }

    template = "ecommerce/detail-view.html"

    return render(request, template, context)

### ecommerce/urls

In [ ]:
from django.conf import settings
from django.contrib import admin
from django.urls import include, path
from ecommerce import views


urlpatterns = [
    path("", views.product_model_list_view, name="Products"),
    path("products/<int:id>/", views.product_model_detail_view, name="ProductDetail"),
]

### templates/ecommerce/detaile-view

In [ ]:
<h1>{{ product.title }}</h1>

<p>Price: {{ product.price }}</p>

## Create View

## ecommerce/views

In [ ]:

from .forms import ProductModelForm

def product_model_create_view(request):
    form = ProductModelForm(request.POST or None)

    if form.is_valid():
        instance = form.save(commit=False)
        instance.save()
        messages.success(request, "Producto creado exitosamente")
        return HttpResponseRedirect("/ecommerce/{product_id}/".format(product_id=instance.id))

    context = {
        "form" : form,
    }

    template = "ecommerce/create-view.html"

    return render(request, template, context)

In [ ]:
from django import forms

from .models import ProductModel

class ProductModelForm(forms.ModelForm):
    class Meta:
        model = ProductModel
        fields = [
            "title",
            "price"
        ]

## templates/ecommerce/messages.html  <-------- esto lo sacamos de la docu de django

In [ ]:
{% if messages %}
<ul class="messages">
    {% for message in messages %}
    <li{% if message.tags %} class="{{ message.tags }}"{% endif %}>{{ message }}</li>
    {% endfor %}
</ul>
{% endif %}

## templates/ecommerce/create-view.html

In [ ]:
{% include "ecommerce/messages.html" %} <--- esto va en todos los templates
<h1>Crear Producto</h1>

<form method = "POST">
    {% csrf_token %}
    {{ form.as_p }}
    <input type="submit" value="Crear Producto">
</form>

## ecommerce/urls

In [ ]:
from django.conf import settings
from django.contrib import admin
from django.urls import include, path
from ecommerce import views


urlpatterns = [
    path("", views.product_model_list_view, name="Products"),
    path("products/<int:id>/", views.product_model_detail_view, name="ProductDetail"),
    path("products/create/", views.product_model_create_view, name="ProductCreate"),
]

## Vista de actualizacion

In [ ]:
def product_model_update_view(request, product_id = None):
    instance = get_object_or_404(ProductModel, id=product_id)
    form = ProductModelForm(request.POST or None, instance=instance)

    if form.is_valid():
        instance = form.save(commit=False)
        print(form.cleaned_data)
        instance.save()
        messages.success(request, "Producto actualizado exitosamente")
        return HttpResponseRedirect("/ecommerce/{product_id}/".format(product_id=instance.id))

    context = {
        "form" : form,
    }

    template = "ecommerce/update-view.html"

    return render(request, template, context)

### templates/update-view.html

In [ ]:
{% include "ecommerce/messages.html" %}
<h1>Actualizando Producto {{form.instance.title}}</h1>

{{form.instance.title}}

<form method = "POST" action=".">
    {% csrf_token %}
    {{ form.as_p }}
    <input type="submit" value="Actualizar Producto">
</form>

### ecommerce/urls

In [ ]:
from django.conf import settings
from django.contrib import admin
from django.urls import include, path
from ecommerce import views


urlpatterns = [
    path("", views.product_model_list_view, name="Products"),
    path("products/<int:id>/", views.product_model_detail_view, name="ProductDetail"),
    path("products/create/", views.product_model_create_view, name="ProductCreate"),
    path("products/<int:product_id>/update/", views.product_model_update_view, name="ProductUpdate"),
]

## templates/delete-view.html

In [ ]:
{% include "ecommerce/messages.html" %}
<h1>Borrando Producto {{product.title}}</h1>
<p>¿Estás seguro de que quieres borrar el producto "{{ product.title }}"?</p>

<form method = "POST" action="">
    {% csrf_token %}
    <input type="submit" value="Borrar Producto">
    <a href="{% url 'Products' %}">Cancelar</a>
</form>

## ecommerce/urls

In [ ]:
from django.conf import settings
from django.contrib import admin
from django.urls import include, path
from ecommerce import views


urlpatterns = [
    path("", views.product_model_list_view, name="Products"),
    path("products/<int:id>/", views.product_model_detail_view, name="ProductDetail"),
    path("products/create/", views.product_model_create_view, name="ProductCreate"),
    path("products/<int:product_id>/update/", views.product_model_update_view, name="ProductUpdate"),
    path("products/<int:product_id>/delete/", views.product_model_delete_view, name="ProductDelete"),
]